In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [2]:
BASE_DIR = Path("..").resolve()

data_file = (
    BASE_DIR
    / "datasets"
    / "amez_dataset_cleaned.xlsx"
)

elasticity_file = (
    BASE_DIR
    / "datasets"
    / "elasticity_features.xlsx"
)

model_folder = (
    BASE_DIR
    / "backend"
    / "ml"
    / "pricing"
    / "models"
)

model_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:")
print(BASE_DIR)

print("\nMain dataset:")
print(data_file)

print("\nElasticity dataset:")
print(elasticity_file)

print("\nModel directory:")
print(model_folder)

Project directory:
C:\Users\RSCH\Desktop\DWD\Final project\Retail-Intelligence-Platform

Main dataset:
C:\Users\RSCH\Desktop\DWD\Final project\Retail-Intelligence-Platform\datasets\amez_dataset_cleaned.xlsx

Elasticity dataset:
C:\Users\RSCH\Desktop\DWD\Final project\Retail-Intelligence-Platform\datasets\elasticity_features.xlsx

Model directory:
C:\Users\RSCH\Desktop\DWD\Final project\Retail-Intelligence-Platform\backend\ml\pricing\models


In [3]:
data = pd.read_excel(
    data_file,
    sheet_name="Overall_Transactions"
)

print("Dataset shape:", data.shape)

data.head()

Dataset shape: (11992, 14)


,InvoiceID,InvoiceDate,CustomerID,ProductID,ProductName,Category,SubCategory,Quantity,UnitPrice,Discount,TotalAmount,StockQuantity,StoreID,StoreName
0,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,8e4669ab-db13-4000-0bd7-f03200008e46,Merino Wool Minimalist Crewneck Knitwear,Fashion & Apparel,Men's Fashion,1,11500.0,2000,9500.0,75,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
1,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,93c93477-c69c-4000-0a0f-adee000093c9,Bio-Peptide Restorative Night Crème,Beauty & Wellness,Skincare Essentials,2,6800.0,2000,11600.0,115,9fbc0fe2-cae9-4000-0ac8-967b00009fbc,Glow Naturals Cosmetics
2,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,7ec257a3-2b97-4000-04df-ce3a00007ec2,Velocity Pro Carbon Running Sneakers,Fashion & Apparel,Footwear,2,17500.0,4000,31000.0,95,a0bc1175-f5e9-4000-0938-88ec0000a0bc,Velocity Athletic Gear
3,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,81c25c5c-d497-4000-0b2f-c5c5000081c2,Ergonomic Breathable Mesh Desk Chair,Home & Living,Modern Furniture,1,28000.0,4000,24000.0,50,99bc0670-cce9-4000-0aa8-9fe9000099bc,Nordic Living Essentials
4,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,93c93477-c69c-4000-0a0f-adee000093c9,Bio-Peptide Restorative Night Crème,Beauty & Wellness,Skincare Essentials,1,6800.0,1000,5800.0,115,9fbc0fe2-cae9-4000-0ac8-967b00009fbc,Glow Naturals Cosmetics


In [4]:
print(data.columns.tolist())

['InvoiceID', 'InvoiceDate', 'CustomerID', 'ProductID', 'ProductName', 'Category', 'SubCategory', 'Quantity', 'UnitPrice', 'Discount', 'TotalAmount', 'StockQuantity', 'StoreID', 'StoreName']


In [5]:
data["InvoiceDate"] = pd.to_datetime(
    data["InvoiceDate"],
    errors="coerce"
)

print(
    "Missing dates:",
    data["InvoiceDate"].isna().sum()
) 
data = data.dropna(
    subset=["InvoiceDate"]
).copy()

Missing dates: 0


In [6]:
numeric_columns = [
    "Quantity",
    "UnitPrice",
    "Discount",
    "StockQuantity"
]

for column in numeric_columns:
    
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

In [7]:
data = data.dropna(
    subset=["Quantity"]
).copy()

data = data[
    data["Quantity"] > 0
].copy()

print(
    "Dataset after cleaning:",
    data.shape
)

Dataset after cleaning: (11992, 14)


In [8]:
data = data.sort_values(
    [
        "ProductID",
        "InvoiceDate"
    ]
).reset_index(drop=True)

data.head()

,InvoiceID,InvoiceDate,CustomerID,ProductID,ProductName,Category,SubCategory,Quantity,UnitPrice,Discount,TotalAmount,StockQuantity,StoreID,StoreName
0,INV-2026-1155,2026-01-15 05:51:38,93d830f9-c5f8-4d2e-9d84-775324d0bc74,0128933d-9879-4346-aafc-a8045c36b087,Minimalist Wool Blend Coat,Fashion & Apparel,Women's Collection,2,34267.0,500,68034.0,173,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
1,INV-2026-1168,2026-01-16 06:21:19,e281776c-e582-4fd9-beb3-e7e616be4308,0128933d-9879-4346-aafc-a8045c36b087,Minimalist Wool Blend Coat,Fashion & Apparel,Women's Collection,4,34748.0,0,138992.0,169,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
2,INV-2026-1172,2026-01-16 06:30:27,09874833-ca5b-4ad4-ad47-8e0b32216f4b,0128933d-9879-4346-aafc-a8045c36b087,Minimalist Wool Blend Coat,Fashion & Apparel,Women's Collection,2,32309.0,2000,62618.0,167,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
3,INV-2026-1183,2026-01-17 06:55:34,f97c586b-abbf-4024-9b3f-45c994b838bd,0128933d-9879-4346-aafc-a8045c36b087,Minimalist Wool Blend Coat,Fashion & Apparel,Women's Collection,2,34283.0,0,68566.0,165,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
4,INV-2026-1208,2026-01-19 07:52:39,dc6fcdb2-4e7a-4a35-b943-d73d51fb119e,0128933d-9879-4346-aafc-a8045c36b087,Minimalist Wool Blend Coat,Fashion & Apparel,Women's Collection,3,33196.0,4000,95588.0,162,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio


In [9]:
data["EffectivePrice"] = (
    data["UnitPrice"] -
    data["Discount"]
)
data[
    [
        "UnitPrice",
        "Discount",
        "EffectivePrice"
    ]
].head(10)

,UnitPrice,Discount,EffectivePrice
0,34267.0,500,33767.0
1,34748.0,0,34748.0
2,32309.0,2000,30309.0
3,34283.0,0,34283.0
4,33196.0,4000,29196.0
5,34088.0,1000,33088.0
6,35105.0,1000,34105.0
7,31410.0,2500,28910.0
8,35280.0,1500,33780.0
9,33972.0,1000,32972.0


In [10]:
data["DayOfWeek"] = (
    data["InvoiceDate"].dt.dayofweek
)

data["Month"] = (
    data["InvoiceDate"].dt.month
)

data["Day"] = (
    data["InvoiceDate"].dt.day
)

In [11]:
elasticity_data = pd.read_excel(
    elasticity_file
)

print(
    "Elasticity dataset:",
    elasticity_data.shape
)

elasticity_data.head()

Elasticity dataset: (11992, 8)


,ProductID,InvoiceDate,Quantity,UnitPrice,Discount,EffectivePrice,PriceChange,DemandChange
0,0128933d-9879-4346-aafc-a8045c36b087,2026-01-15 05:51:38,2,34267.0,500,34017.000000,NaN,NaN
1,0128933d-9879-4346-aafc-a8045c36b087,2026-01-16 06:21:19,4,34748.0,0,34748.000000,0.021489,1.0
2,0128933d-9879-4346-aafc-a8045c36b087,2026-01-16 06:30:27,2,32309.0,2000,31309.000000,-0.098970,-0.5
3,0128933d-9879-4346-aafc-a8045c36b087,2026-01-17 06:55:34,2,34283.0,0,34283.000000,0.094989,0.0
4,0128933d-9879-4346-aafc-a8045c36b087,2026-01-19 07:52:39,3,33196.0,4000,31862.666667,-0.070599,0.5


In [12]:
elasticity_data["EffectivePrice"] = pd.to_numeric(
    elasticity_data["EffectivePrice"],
    errors="coerce"
)

elasticity_data["Quantity"] = pd.to_numeric(
    elasticity_data["Quantity"],
    errors="coerce"
)

elasticity_data = elasticity_data.dropna(
    subset=[
        "ProductID",
        "EffectivePrice",
        "Quantity"
    ]
).copy()

elasticity_data = elasticity_data[
    (elasticity_data["EffectivePrice"] > 0) &
    (elasticity_data["Quantity"] > 0)
].copy()

In [13]:
price_demand = (
    elasticity_data
    .groupby(
        [
            "ProductID",
            "EffectivePrice"
        ],
        as_index=False
    )
    .agg(
        TotalDemand=("Quantity", "sum"),
        SalesObservations=("InvoiceDate", "count")
    )
)

price_demand.head()

,ProductID,EffectivePrice,TotalDemand,SalesObservations
0,0128933d-9879-4346-aafc-a8045c36b087,29113.0,1,1
1,0128933d-9879-4346-aafc-a8045c36b087,29117.0,1,1
2,0128933d-9879-4346-aafc-a8045c36b087,30027.0,2,1
3,0128933d-9879-4346-aafc-a8045c36b087,30160.0,2,1
4,0128933d-9879-4346-aafc-a8045c36b087,30184.0,2,1


In [14]:
def estimate_elasticity(product_data):
    
    product_data = product_data[
        (product_data["EffectivePrice"] > 0) &
        (product_data["TotalDemand"] > 0)
    ].copy()

    product_data = product_data.drop_duplicates(
        subset=["EffectivePrice"]
    )

    if len(product_data) < 3:
        return None

    log_price = np.log(
        product_data["EffectivePrice"]
    )

    log_demand = np.log(
        product_data["TotalDemand"]
    )

    slope = np.polyfit(
        log_price,
        log_demand,
        1
    )[0]

    return float(slope)

In [15]:
elasticity_results = []

for product_id, group in price_demand.groupby(
    "ProductID"
):
    
    elasticity = estimate_elasticity(
        group
    )

    elasticity_results.append({
        "ProductID": product_id,
        "Elasticity": elasticity
    })

elasticity_df = pd.DataFrame(
    elasticity_results
)

print(
    "Elasticity estimates:",
    len(elasticity_df)
)

elasticity_df.head()

Elasticity estimates: 121


,ProductID,Elasticity
0,0128933d-9879-4346-aafc-a8045c36b087,3.746624
1,03df1a4a-b5a2-40ee-b621-f064b782d4a1,1.676465
2,04df7b21-ab56-460d-96c8-25aed6e602a0,2.169678
3,06a13e3b-0b08-40f7-bc7c-74fc8f197b7f,2.217897
4,0a7982e2-2a56-4dea-b738-821ef43ca9ae,2.750905


In [16]:
data = data.merge(
    elasticity_df,
    on="ProductID",
    how="left"
)

print(
    "Dataset shape:",
    data.shape
)

Dataset shape: (11992, 19)


In [17]:
elasticity_median = data[
    "Elasticity"
].median()

print(
    "Median elasticity:",
    elasticity_median
)
data["Elasticity"] = (
    data["Elasticity"]
    .fillna(elasticity_median)
)
print(
    "Missing elasticity:",
    data["Elasticity"].isna().sum()
)

Median elasticity: 2.712063595554656
Missing elasticity: 0


In [18]:
data["PreviousDemand"] = (
    data
    .groupby("ProductID")["Quantity"]
    .shift(1)
)

In [19]:
data["PreviousDemand2"] = (
    data
    .groupby("ProductID")["Quantity"]
    .shift(2)
)

In [20]:
data["RollingDemandMean"] = (
    data
    .groupby("ProductID")["Quantity"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )
)

In [21]:
data["HistoricalMaxDemand"] = (
    data
    .groupby("ProductID")["Quantity"]
    .transform(
        lambda x:
        x.shift(1)
        .expanding()
        .max()
    )
)

In [22]:
data["HistoricalAvgDemand"] = (
    data
    .groupby("ProductID")["Quantity"]
    .transform(
        lambda x:
        x.shift(1)
        .expanding()
        .mean()
    )
)

In [23]:
data[
    [
        "ProductID",
        "InvoiceDate",
        "Quantity",
        "PreviousDemand",
        "PreviousDemand2",
        "RollingDemandMean",
        "HistoricalMaxDemand",
        "HistoricalAvgDemand"
    ]
].head(20)

,ProductID,InvoiceDate,Quantity,PreviousDemand,PreviousDemand2,RollingDemandMean,HistoricalMaxDemand,HistoricalAvgDemand
0,0128933d-9879-4346-aafc-a8045c36b087,2026-01-15 05:51:38,2,NaN,NaN,NaN,NaN,NaN
1,0128933d-9879-4346-aafc-a8045c36b087,2026-01-16 06:21:19,4,2.0,NaN,2.000000,2.0,2.000000
2,0128933d-9879-4346-aafc-a8045c36b087,2026-01-16 06:30:27,2,4.0,2.0,3.000000,4.0,3.000000
3,0128933d-9879-4346-aafc-a8045c36b087,2026-01-17 06:55:34,2,2.0,4.0,2.666667,4.0,2.666667
4,0128933d-9879-4346-aafc-a8045c36b087,2026-01-19 07:52:39,3,2.0,2.0,2.666667,4.0,2.500000
5,0128933d-9879-4346-aafc-a8045c36b087,2026-01-24 10:05:05,3,3.0,2.0,2.333333,4.0,2.600000
6,0128933d-9879-4346-aafc-a8045c36b087,2026-01-25 10:23:21,2,3.0,3.0,2.666667,4.0,2.666667
7,0128933d-9879-4346-aafc-a8045c36b087,2026-01-25 10:30:12,2,2.0,3.0,2.666667,4.0,2.571429
8,0128933d-9879-4346-aafc-a8045c36b087,2026-01-26 10:53:02,2,2.0,2.0,2.333333,4.0,2.500000
9,0128933d-9879-4346-aafc-a8045c36b087,2026-01-28 11:38:42,2,2.0,2.0,2.000000,4.0,2.444444


In [24]:
model_data = data.dropna(
    subset=[
        "PreviousDemand",
        "PreviousDemand2",
        "RollingDemandMean",
        "HistoricalMaxDemand",
        "HistoricalAvgDemand"
    ]
).copy()

print(
    "Model dataset shape:",
    model_data.shape
)

Model dataset shape: (11750, 24)


In [25]:
print(
    model_data["Quantity"].describe()
)

count    11750.000000
mean         2.363574
std          1.242179
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          5.000000
Name: Quantity, dtype: float64


In [26]:
features = [
    "EffectivePrice",
    "Discount",
    "DayOfWeek",
    "Month",
    "Day",
    "StockQuantity",
    "Elasticity",
    "PreviousDemand",
    "PreviousDemand2",
    "RollingDemandMean",
    "HistoricalMaxDemand",
    "HistoricalAvgDemand"
]
missing_features = [
    feature
    for feature in features
    if feature not in model_data.columns
]

print(
    "Missing features:",
    missing_features
)

Missing features: []


In [27]:
X = model_data[
    features
].copy()

y = model_data[
    "Quantity"
].copy()

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

X shape: (11750, 12)
y shape: (11750,)


In [28]:
print(
    X.isnull().sum()
)
print(
    "\nTarget missing values:",
    y.isnull().sum()
)

EffectivePrice         0
Discount               0
DayOfWeek              0
Month                  0
Day                    0
StockQuantity          0
Elasticity             0
PreviousDemand         0
PreviousDemand2        0
RollingDemandMean      0
HistoricalMaxDemand    0
HistoricalAvgDemand    0
dtype: int64

Target missing values: 0


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)

Training rows: 9400
Testing rows: 2350


In [30]:
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model.fit(
    X_train,
    y_train
)

print(
    "LightGBM training completed."
)

LightGBM training completed.


In [31]:
y_pred = model.predict(
    X_test
)

print(
    y_pred[:10]
)

[2.79062885 2.29551725 2.63612023 2.06962759 1.05577022 2.68363232
 2.47770045 2.46514463 2.59600692 2.8583597 ]


In [32]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print(
    "MAE:",
    round(mae, 3)
)

print(
    "RMSE:",
    round(rmse, 3)
)

print(
    "R²:",
    round(r2, 3)
)

MAE: 0.98
RMSE: 1.206
R²: 0.079
